# Quickstart: ask over six passages

Run from the repository root on a CUDA GPU. On Kaggle, attach the project bundle and the Tiny Aya Global model first. Outside Kaggle, accept the base-model terms on Hugging Face and authenticate using your own account. This notebook has not yet been GPU-validated.

In [ ]:
%pip install --no-deps -r requirements-gpu.lock.txt
%pip install --no-deps -e .

In [ ]:
import json
import sqlite3
from dataclasses import asdict
from pathlib import Path

from scripts.ask import load_model_for_system
from ragharness import system

docs = [
    {'title': 'Returns', 'text': 'Nimbus Home accepts returns within 30 days after delivery.'},
    {'title': 'Warranty', 'text': 'Nimbus Home devices have a two-year warranty.'},
    {'title': 'Shipping', 'text': 'Orders ship within three business days.'},
    {'title': 'Support', 'text': 'Support is available Monday through Friday.'},
    {'title': 'Packaging', 'text': 'Keep the original packaging when returning a device.'},
    {'title': 'Delivery', 'text': 'A tracking link is sent when an order ships.'},
]
model, tokenizer = load_model_for_system('BMEscanor/against-the-safety-tiny-aya-rag-main')
# Fail loudly if this Python build loses FTS5 after the model libraries load.
with sqlite3.connect(':memory:') as connection:
    connection.execute('CREATE VIRTUAL TABLE probe USING fts5(text)')
backend, _ = system.index_documents(docs, Path('runs/quickstart/docs.sqlite'))
rag = system.RagSystem(model, tokenizer, backend)


The next three cells ask an answerable question, one missing from the documents, and a two-part question. The actual decision depends on the model; a refusal on the two-part question is a valid result to inspect.

In [ ]:
decision = rag.ask('What is the return window for Nimbus Home devices?')
print(json.dumps(asdict(decision), ensure_ascii=False, indent=2))

In [ ]:
decision = rag.ask('What is the price of a Nimbus Home device?')
print(json.dumps(asdict(decision), ensure_ascii=False, indent=2))

In [ ]:
decision = rag.ask('What are the return window and the warranty length for Nimbus Home devices?')
print(json.dumps(asdict(decision), ensure_ascii=False, indent=2))